# 3. Rekurzió. Tetszőleges számú pozícionális és név szerinti paraméter

Ez a notebook a következő témákat tárgyalja:

- Függvény definiálása másik függvény törzsében (*nested function*)
- Rekurzív függvények (*recursion*): faktoriális, Fibonacci-számok
- Iterálható objektumok kicsomagolása (*unpacking*)
- Változó számú pozicionális paraméter (`*args`)
- Változó számú név szerinti paraméter (`**kwargs`)

---

## 3.1 Függvény definiálása másik függvény törzsében

Az előző notebookban két függvényt definiáltunk: `paros_e()` és `hullamzo()`. A `hullamzo()` a `paros_e()` függvényt a **globális hatókörből** éri el. Ez azt jelenti, hogy `paros_e()` bárki számára elérhető és hívható -- holott kizárólag a `hullamzo()` használja.

### Előtte: `paros_e()` globális szinten

In [ ]:
def paros_e(szam: int) -> bool:
    """Visszaadja, hogy a szám páros-e."""
    return szam % 2 == 0


def hullamzo(mondat: str) -> str:
    """Visszaadja a szöveg hullámzó formáját."""
    betuk = []
    for ch in mondat:
        if ch in " ,.;":
            continue
        betuk.append(ch)
    for i in range(len(betuk)):
        if paros_e(i):  # globális hatókörből éri el
            betuk[i] = betuk[i].upper()
        else:
            betuk[i] = betuk[i].lower()
    return "".join(betuk)


print(hullamzo("Geza kek az eg."))
print(paros_e(4))  # kívülről is hívható -- de csak a hullamzo() használja

### Utána: `paros_e()` beágyazva a `hullamzo()` törzsébe

A `paros_e()` definícióját áthelyezzük a `hullamzo()` függvény **belsejébe**. Így `paros_e()` **belső függvénnyé** (*nested function*) válik: kívülről nem hívható, a `hullamzo()` implementációs részlete marad.

In [ ]:
def hullamzo_v2(mondat: str) -> str:
    """Visszaadja a szöveg hullámzó formáját. A paros_e() belső függvény."""

    def paros_e(szam: int) -> bool:
        """Visszaadja, hogy a szám páros-e."""
        return szam % 2 == 0

    betuk = []
    for ch in mondat:
        if ch in " ,.;":
            continue
        betuk.append(ch)
    for i in range(len(betuk)):
        if paros_e(i):  # a belső függvényt hívja
            betuk[i] = betuk[i].upper()
        else:
            betuk[i] = betuk[i].lower()
    return "".join(betuk)


print(hullamzo_v2("Geza kek az eg."))

In [ ]:
# A belső paros_e() kívülről NEM hívható:
try:
    hullamzo_v2.paros_e(4)
except AttributeError as hiba:
    print(f"AttributeError: {hiba}")

### Miért hasznos a beágyazás?

- A belső függvény a külső függvény **implementációs részlete** -- nem szennyezi a globális névteret.
- Kívülről nem hívható, így a modul felhasználóját nem terheli felesleges függvényekkel.
- A LEGB szabály **Enclosing (E)** hatóköre teszi lehetővé: a belső függvény hozzáfér a külső függvény lokális változóihoz. Jelen esetben a `paros_e()` a `hullamzo_v2()` lokális hatókörében él.

---

## 3.2 Rekurzív függvények

A **rekurzió** (*recursion*) olyan programozási technika, amelyben egy függvény **önmagát hívja meg**. Minden rekurzív függvénynek két része van: a **megállási feltétel** (*base case*), amely megadja, mikor kell abbahagyni a hívást, és a **rekurzív eset** (*recursive case*), amely a problémát kisebb részproblémára bontja, és újra meghívja a függvényt.

### 3.2.1 Faktoriális

**Matematikai képlet:**

$$n! = n \times (n-1)!$$

Megállási feltétel: $0! = 1$

Példa: $4! = 4 \times 3! = 4 \times 3 \times 2! = 4 \times 3 \times 2 \times 1! = 4 \times 3 \times 2 \times 1 \times 0! = 24$

In [ ]:
def faktorialis(n: int, melyseg: int = 0) -> int:
    """Kiszámítja n faktoriálisát rekurzívan. A melyseg paraméter a behúzást vezérli."""
    behuzas = "  " * melyseg
    print(f"{behuzas}Függvényhívás: faktorialis({n})")

    if n == 0:
        # megállási feltétel
        print(f"{behuzas}Visszatérés: faktorialis(0) = 1")
        return 1

     
    eredmeny = n * faktorialis(n - 1, melyseg + 1)
    print(f"{behuzas}Visszatérés: faktorialis({n}) = {eredmeny}")
    return eredmeny


faktorialis(4)

A `print()` hívások szemléltetik a **hívási láncot**: a függvény egyre mélyebbre hív, amíg el nem éri a megállási feltételt (`n == 0`), majd a visszatérési értékek felfelé terjednek.

**Mermaid diagram** -- a `faktorialis(4)` hívási lánca:

```mermaid
graph TB
    A["faktorialis(4)"] -->|"n=3"| B["faktorialis(3)"]
    B -->|"n=2"| C["faktorialis(2)"]
    C -->|"n=1"| D["faktorialis(1)"]
    D -->|"n=0"| E["faktorialis(0)"]
    E -.->|"return 1"| D
    D -.->|"return 1"| C
    C -.->|"return 2"| B
    B -.->|"return 6"| A
    A -.->|"return 24"| F(("eredmény: 24"))
```

A folytonos élek a hívásokat, a szaggatott élek a visszatérési értékeket jelölik.

### 3.2.2 Fibonacci-számok

**Képlet:**

$$F(0) = 0, \quad F(1) = 1, \quad F(n) = F(n-1) + F(n-2) \quad \text{ha } n \geq 2$$

Az első néhány Fibonacci-szám: 0, 1, 1, 2, 3, 5, 8, 13, 21, ...

In [ ]:
def fibo(n: int) -> int:
    """Visszaadja az n-edik Fibonacci-számot rekurzívan."""
    if n in (0, 1):
        return n
    return fibo(n - 1) + fibo(n - 2)


print(f"fibo(6) = {fibo(6)}")

**Mermaid diagram** -- a `fibo(6)` teljes rekurzív hívási fája:

```mermaid
graph TB
    F6["fibo(6)"] --> F5["fibo(5)"]
    F6 --> F4a["fibo(4)"]

    F5 --> F4b["fibo(4)"]
    F5 --> F3a["fibo(3)"]

    F4b --> F3b["fibo(3)"]
    F4b --> F2a["fibo(2)"]

    F4a --> F3c["fibo(3)"]
    F4a --> F2b["fibo(2)"]

    F3a --> F2c["fibo(2)"]
    F3a --> F1a["fibo(1) = 1"]

    F3b --> F2d["fibo(2)"]
    F3b --> F1b["fibo(1) = 1"]

    F3c --> F2e["fibo(2)"]
    F3c --> F1c["fibo(1) = 1"]

    F2a --> F1d["fibo(1) = 1"]
    F2a --> F0a["fibo(0) = 0"]

    F2b --> F1e["fibo(1) = 1"]
    F2b --> F0b["fibo(0) = 0"]

    F2c --> F1f["fibo(1) = 1"]
    F2c --> F0c["fibo(0) = 0"]

    F2d --> F1g["fibo(1) = 1"]
    F2d --> F0d["fibo(0) = 0"]

    F2e --> F1h["fibo(1) = 1"]
    F2e --> F0e["fibo(0) = 0"]
```

A diagram jól mutatja a probléma lényegét: **azonos részproblémák többszöri kiszámítása**. Például a `fibo(3)` háromszor, a `fibo(2)` ötször kerül kiszámításra. Ez exponenciálisan növekvő futásidőt eredményez.

#### Optimalizálás: `cache` dekorátor

A `functools.cache` dekorátor **megjegyzi** (*memoization*) a már kiszámított értékeket, így minden `fibo(k)` hívás legfeljebb egyszer fut le ténylegesen.

In [ ]:
hivasok_szama = 0


def fibo_lassu(n: int) -> int:
    """Fibonacci cache nélkül."""
    global hivasok_szama
    hivasok_szama += 1
    if n == 0:
        return 0
    if n == 1:
        return 1
    return fibo_lassu(n - 1) + fibo_lassu(n - 2)


hivasok_szama = 0
fibo_lassu(20)
print(f"Cache NÉLKÜL -- fibo(20) hívások száma: {hivasok_szama}")

In [ ]:
from functools import cache

cache_hivasok_szama = 0


@cache
def fibo_gyors(n: int) -> int:
    """Fibonacci cache dekorátorral."""
    global cache_hivasok_szama
    cache_hivasok_szama += 1
    if n == 0:
        return 0
    if n == 1:
        return 1
    return fibo_gyors(n - 1) + fibo_gyors(n - 2)


cache_hivasok_szama = 0
fibo_gyors(20)
print(f"Cache-sel -- fibo(20) hívások száma:   {cache_hivasok_szama}")

A `@cache` dekorátorral a hívások száma **lineárissá** csökken: `fibo(20)` mindössze 21 tényleges hívást igényel a korábbi több ezernyi helyett. A dekorátorokról egy későbbi notebookban lesz szó részletesen.

### 3.2.3 Gyakorló feladatok

#### Feladat 1 -- Rekurzív számsorozat

Adott a következő sorozat:

$$x(n) = (x(n-1))^2 - 2 \cdot x(n-2) + 3$$

Megállási feltételek: $x(0) = -2$, $x(1) = 3$.

Írj egy `def sorozat(n: int) -> int` függvényt, amely visszaadja $x(n)$ értékét.

**Tesztek:**
- `sorozat(0)` -> `-2`
- `sorozat(1)` -> `3`
- `sorozat(2)` -> `16`
- `sorozat(5)` -> `4093439897`

In [ ]:
def sorozat(n: int) -> int:
    """Visszaadja az x(n) értékét a megadott rekurzív sorozatból."""
    pass


# assert sorozat(0) == -2
# assert sorozat(1) == 3
# assert sorozat(2) == 16
# assert sorozat(5) == 4093439897
# print("Minden teszt sikeres!")

#### Feladat 2 -- Bináris keresés rekurzióval

A **bináris keresés** (*binary search*) egy rendezett listában keres meg egy elemet úgy, hogy minden lépésben **felezi** a keresési tartományt:

1. Kiszámítja a középső elem indexét.
2. Ha a középső elem egyenlő a keresett értékkel -> megtalálta.
3. Ha a keresett érték kisebb -> a bal felében keres tovább.
4. Ha a keresett érték nagyobb -> a jobb felében keres tovább.
5. Ha a tartomány üres (`bal > jobb`) -> az elem nincs a listában.

Írj egy rekurzív függvényt:

```python
def binaris_kereses(lista: list[int], x: int, bal: int = 0, jobb: int | None = None) -> int | None
```

A függvény visszaadja `x` indexét, vagy `None`-t, ha nincs a listában.

In [ ]:
def binaris_kereses(lista: list[int], x: int, bal: int = 0, jobb: int | None = None) -> int | None:
    """Rekurzív bináris keresés. Visszaadja x indexét vagy None-t."""
    pass


# rendezett_lista = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
# assert binaris_kereses(rendezett_lista, 23) == 5
# assert binaris_kereses(rendezett_lista, 2) == 0
# assert binaris_kereses(rendezett_lista, 91) == 9
# assert binaris_kereses(rendezett_lista, 50) is None
# assert binaris_kereses([], 10) is None
# print("Minden teszt sikeres!")

---

## 3.3 Iterálható objektumok kicsomagolása (*unpacking*)

Pythonban egy iterálható objektum (tuple, lista stb.) elemeit egyszerre, **párhuzamosan** rendelhetjük több változóhoz. Ezt nevezzük **kicsomagolásnak** (*unpacking*).

### Alap kicsomagolás

A bal oldalon annyi változó áll, ahány elem van a jobb oldalon.

In [ ]:
a, b, c = 10, 20, 30

print(f"a = {a}")
print(f"b = {b}")
print(f"c = {c}")

In [ ]:
# Listából is működik:
szamok = [40, 50, 60]
d, e, f = szamok

print(f"d = {d}, e = {e}, f = {f}")

### A `*` (csillag / *star*) operátor: maradék elemek összegyűjtése

Ha nem tudjuk előre, hány elem lesz, a `*` operátorral a **maradék elemeket** egy listába gyűjthetjük.

In [ ]:
a, b, *c = 10, 20, 30, 40

print(f"a = {a}")     # 10
print(f"b = {b}")     # 20
print(f"c = {c}")     # [30, 40] -- lista!

e, f, *g = (1, 2, 3, 4, 5)
print(f"e = {e}")
print(f"f = {f}")
print(f"g = {g}, típusa: {type(g)}") akkor is lista lesz az eredmény ha tuple-ból csomagolunk ki


### A `*` operátor középen

A `*` operátor nem csak a végén állhat -- tetszőleges pozícióban elhelyezhető.

In [ ]:
a, *b, c = 10, 20, 30, 40, 50

print(f"a = {a}")     # 10
print(f"b = {b}")     # [20, 30, 40]
print(f"c = {c}")     # 50

In [ ]:
# Az elején is állhat:
*a, b, c = 10, 20, 30, 40, 50

print(f"a = {a}")     # [10, 20, 30]
print(f"b = {b}")     # 40
print(f"c = {c}")     # 50

### Hibás példák

Nem minden kicsomagolás érvényes. Az alábbiak `ValueError`-t vagy `SyntaxError`-t dobnak.

In [ ]:
# Hiba 1: túl kevés érték a változók számához képest
try:
    a, b, c = 10, 20
except ValueError as hiba:
    print(f"ValueError: {hiba}")

In [ ]:
# Hiba 2: túl sok érték a változók számához képest
try:
    a, b = 10, 20, 30
except ValueError as hiba:
    print(f"ValueError: {hiba}")

In [ ]:
# Hiba 3: két * operátor -- ez SyntaxError, ezért nem futtatható try/except-ben.
# Az alábbi kód szintaktikailag hibás:
#
# *a, *b = 10, 20, 30, 40
#
# SyntaxError: multiple starred expressions in assignment
print("Két * operátor egy értékadásban SyntaxError-t okoz.")

---

## 3.4 Változó számú pozicionális paraméter (`*args`)

A `*args` paraméter lehetővé teszi, hogy egy függvény **tetszőleges számú pozicionális argumentumot** fogadjon el. A függvény törzsében az `args` egy **tuple**-ként érhető el, amely tartalmazza az összes átadott értéket.

### Példa 1 -- Számok összegzése

In [ ]:
def osszegzes(*szamok) -> int:
    """Összeadja a kapott számokat. Nem kötelező a paraméter *args-ként megadni."""
    print(f"Kapott argumentumok: {szamok} (típus: {type(szamok).__name__})")
    eredmeny = 0
    for szam in szamok:
        eredmeny += szam
    return eredmeny


print(osszegzes(10, 20, 30))
print(osszegzes(10, 20, 30, 40, 50))
print(osszegzes(10))
print(osszegzes())  # 0 argumentum is megengedett

### Példa 2 -- Évszakok és hónapok

In [ ]:
def evszak_honapjai(evszak_neve: str, *honapok) -> None:
    """Kilistázza egy évszak hónapjait."""
    print(f"\n{evszak_neve} hónapjai:")
    for i, honap in enumerate(honapok, start=1):
        print(f"  {i}. {honap}")


evszak_honapjai("Tavasz", "március", "április", "május")
evszak_honapjai("Nyár", "június", "július", "augusztus")
evszak_honapjai("Ősz", "szeptember", "október", "november")
evszak_honapjai("Tél", "december", "január", "február")

Az első paraméter (`evszak_neve`) egy hagyományos pozicionális paraméter; a többi argumentum a `*honapok` tuple-be kerül.

### Hibás példa -- `*args` utáni pozicionális paraméter

In [ ]:
# Ha *args utáni paraméterekhez, argumentumot azt név-kulcsszó szerint lehet megadni.
# Pozicionálisan nem lehet, mert az *args mindent "elnyel".

def hibas_pelda(*szamok, szorzo):
    """A szorzo paraméterhez argumentumot csak kulcsszóval adható meg."""
    return [sz * szorzo for sz in szamok]


# Ez működik -- szorzo kulcsszóval megadva:
print(hibas_pelda(10, 20, 30, szorzo=2))

# Ez hibát dob -- a 2-t is az *args kapja meg:
try:
    hibas_pelda(10, 20, 30, 2)
except TypeError as hiba:
    print(f"TypeError: {hiba}")

---

## 3.5 Változó számú név szerinti paraméter (`**kwargs`)

A `**kwargs` paraméter lehetővé teszi, hogy egy függvény **tetszőleges számú kulcsszavas argumentumot** (*keyword argument*) fogadjon. A függvény törzsében a `kwargs` egy **dict**-ként érhető el, ahol a kulcsok a paraméternevek (stringek), az értékek pedig a hozzájuk rendelt argumentumok.

### Példa 1 -- Számítógép-konfiguráció

In [ ]:
def konfiguracio(gep_neve: str, **alkatreszek) -> None:
    """Kiírja a számítógép konfigurációját."""
    print(f"\n--- {gep_neve} konfigurációja ---")
    print(f"Kapott kwargs típusa: {type(alkatreszek).__name__}")
    for tipus, modell in alkatreszek.items():
        print(f"  {tipus}: {modell}")


konfiguracio(
    "GTamer PC",
    gpu="RTX 4090",
    cpu="Ryzen 9 7950X",
    memoria="32GB DDR5",
    ssd="2TB NVMe",
)

konfiguracio(
    "Asztali PC",
    cpu="Intel i7-13400",
    memoria="16GB DDR4",
)

### Példa 2 -- kwargs numerikus értékekkel

In [ ]:
def szamok_osszege(**szamok) -> int:
    """Összeadja a kulcsszavas argumentumok értékeit."""
    print(f"Kapott párok: {szamok}")
    osszeg = 0
    for nev, ertek in szamok.items():
        print(f"  {nev} = {ertek}")
        osszeg += ertek
    return osszeg


eredmeny = szamok_osszege(a=10, b=20, c=30)
print(f"Összeg: {eredmeny}")

### Hibás példa -- `**kwargs` rossz pozícióban

In [ ]:
# A **kwargs-nak mindig az utolsó paraméter kell legyen.
# Ha utána hagyományos paramétert teszünk, SyntaxError-t kapunk:
#
# def hibas(**kwargs, nev):
#     pass
#
# SyntaxError: arguments cannot follow var-keyword argument

print("A **kwargs mindig a paramétersor VÉGÉN kell álljon.")
print("Helyes sorrend: def fv(a, b, *args, **kwargs)")

In [ ]:
# Hibás hívás: pozicionális argumentum megadása kulcsszavasként is
def konfig(nev: str, **beallitasok):
    print(f"{nev}: {beallitasok}")


try:
    konfig("PC", nev="másik")  # 'nev' kétszer megadva
except TypeError as hiba:
    print(f"TypeError: {hiba}")

---

## Megoldások

<details>
<summary><strong>3.2.3 Feladat 1 -- Rekurzív számsorozat</strong></summary>

```python
def sorozat(n: int) -> int:
    "Visszaadja az x(n) értékét a megadott rekurzív sorozatból."    
    
    if n == 0:
        return -2
    if n == 1:
        return 3
    return sorozat(n - 1) ** 2 - 2 * sorozat(n - 2) + 3
```

Ellenőrzés lépésről lépésre:
- `sorozat(0)` -> `-2`
- `sorozat(1)` -> `3`
- `sorozat(2)` -> `3**2 - 2*(-2) + 3 = 9 + 4 + 3 = 16`
- `sorozat(3)` -> `16**2 - 2*3 + 3 = 256 - 6 + 3 = 253`
- `sorozat(4)` -> `253**2 - 2*16 + 3 = 64009 - 32 + 3 = 63980`
- `sorozat(5)` -> `63980**2 - 2*253 + 3 = 4093440400 - 506 + 3 = 4093439897`

</details>

<details>
<summary><strong>3.2.3 Feladat 2 -- Bináris keresés</strong></summary>

```python
def binaris_kereses(lista: list[int], x: int, bal: int = 0, jobb: int | None = None) -> int | None:
    "Rekurzív bináris keresés. Visszaadja x indexét vagy None-t."    
    
    if jobb is None:
        jobb = len(lista) - 1

    if bal > jobb:
        return None

    kozep = (bal + jobb) // 2

    if lista[kozep] == x:
        return kozep
    elif lista[kozep] < x:
        return binaris_kereses(lista, x, kozep + 1, jobb)
    else:
        return binaris_kereses(lista, x, bal, kozep - 1)
```

</details>